In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import pandas_datareader.data as web

In [2]:
#plot 한글 적용
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

### 팩터 데이터 불러오기

In [3]:
START_DATE = '2013-12-01' #단순수익률 산정이 필요하니 조회가 필요한 일자에서 -1Month로 조회
END_DATE = '2018-12-31'

In [12]:
# 3-팩터 
df_three_factor = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start=START_DATE, end=END_DATE)[0]

# 모맨텀 팩터
df_mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start=START_DATE, end=END_DATE)[0] 

# 5-팩터
df_five_factor = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench', start=START_DATE, end=END_DATE)[0]

### 데이터 불러오기(amazon 2014~2018년도)

In [13]:
file_path = "../Data-collection/dailyStock"
target = "AMZN"

amzn_df = pd.read_csv(f"{file_path}/{target}.csv", encoding="utf-8")

amzn_df['Date'] = pd.to_datetime(amzn_df['Date']) # 날짜로 변환
amzn_df = amzn_df.set_index('Date')               # Date를 인덱스로 설정
amzn_df = amzn_df.loc[START_DATE:END_DATE]  # 분석 기간

y= amzn_df['adj_close'].resample("ME").last().pct_change().dropna()

y.index = y.index.to_period('M')
y.name = 'rtn'

### 4-팩터(3-팩터 + 모맨텀)

In [24]:
four_factor_data = df_three_factor.join(df_mom).join(y).dropna()

#컬럼명 변경
four_factor_data.columns = ['mkt', 'smb', 'hml', 'rf', 'mom', 'rtn']

#백분율데이터 소수로 변환
four_factor_data.loc[:, four_factor_data.columns != 'rtn'] /= 100

# 초과수익률
four_factor_data['excess_rtn'] = four_factor_data.rtn - four_factor_data.rf

four_factor_data.head()

,mkt,smb,hml,rf,mom,rtn,excess_rtn
Date,,,,,,,
2014-01,-0.0332,0.0093,-0.0199,0.0,0.0164,-0.100554,-0.100554
2014-02,0.0466,0.0036,-0.0035,0.0,0.0216,0.009507,0.009507
2014-03,0.0043,-0.0186,0.0490,0.0,-0.0327,-0.071058,-0.071058
2014-04,-0.0018,-0.0418,0.0122,0.0,-0.0389,-0.095847,-0.095847
2014-05,0.0205,-0.0185,-0.0010,0.0,0.0088,0.027685,0.027685


In [28]:
four_factor_model = smf.ols(formula='excess_rtn ~ mkt + smb + hml + mom', data=four_factor_data).fit()

print(four_factor_model.summary())

                            OLS Regression Results                            
Dep. Variable:             excess_rtn   R-squared:                       0.550
Model:                            OLS   Adj. R-squared:                  0.517
Method:                 Least Squares   F-statistic:                     16.78
Date:                Fri, 18 Sep 2026   Prob (F-statistic):           4.80e-09
Time:                        20:47:21   Log-Likelihood:                 86.531
No. Observations:                  60   AIC:                            -163.1
Df Residuals:                      55   BIC:                            -152.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0097      0.008      1.210      0.2

### 5-팩터 

In [27]:
five_factor_data = df_five_factor.join(y).dropna()

#컬럼명 변경
five_factor_data.columns = ['mkt', 'smb', 'hml', 'rmw', 'cma', 'rf', 'rtn']

#백분율데이터 소수로 변환
five_factor_data.loc[:, five_factor_data.columns != 'rtn'] /= 100

# 초과수익률
five_factor_data['excess_rtn'] = five_factor_data.rtn - five_factor_data.rf

five_factor_data.head()

,mkt,smb,hml,rmw,cma,rf,rtn,excess_rtn
Date,,,,,,,,
2014-01,-0.0330,0.0060,-0.0199,-0.0391,-0.0154,0.0,-0.100554,-0.100554
2014-02,0.0468,0.0013,-0.0035,-0.0025,-0.0054,0.0,0.009507,0.009507
2014-03,0.0041,-0.0113,0.0490,0.0202,0.0192,0.0,-0.071058,-0.071058
2014-04,-0.0019,-0.0407,0.0122,0.0340,0.0093,0.0,-0.095847,-0.095847
2014-05,0.0204,-0.0186,-0.0010,-0.0009,-0.0096,0.0,0.027685,0.027685


In [29]:
five_factor_model = smf.ols(formula='excess_rtn ~ mkt + smb + hml + rmw + cma', data=five_factor_data).fit()
print(five_factor_model.summary())

                            OLS Regression Results                            
Dep. Variable:             excess_rtn   R-squared:                       0.595
Model:                            OLS   Adj. R-squared:                  0.557
Method:                 Least Squares   F-statistic:                     15.85
Date:                Fri, 18 Sep 2026   Prob (F-statistic):           1.38e-09
Time:                        20:48:02   Log-Likelihood:                 89.697
No. Observations:                  60   AIC:                            -167.4
Df Residuals:                      54   BIC:                            -154.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0101      0.008      1.320      0.1